# Learned meshless FD weights: Test Problem 1

Discretization: the FD solver of `PHS_fd_TP1.m` (fitted interface nodes, one stencil per row, the `nloc` = 3Q nearest nodes of the same side).

**Weights of one stencil** (scaled coordinates $\hat y=(y-x)/r$):
$$\hat w=\hat w_{\rm base}+s\,(I-\Pi)\,v,$$
where $\hat w_{\rm base}$ are the RBF-FD weights (PHS $r^5$ + polynomials of degree `deg`), $\Pi$ is the orthogonal projector onto the range of the polynomial matrix $P$, and $s$ is the rms of $\hat w_{\rm base}$.
Since $P^T(I-\Pi)=0$, every row is exact on the polynomials of degree `deg` for every $v$; no kernel is reproduced. $v=0$ is the baseline.

**Loss** (manufactured solutions on the training node sets; index 0 = baseline):
$$L=\frac{\|E\|^2}{\|E_0\|^2}+\lambda_\tau\,\frac12\sum_{g}\frac{\|\tau_g\|^2}{\|\tau_{0,g}\|^2}+\lambda_{1}\,\frac{\rho}{\rho_0},$$
- $E=-A^{-1}\tau$: error of the discrete solution (the discretized PDE, with its interface conditions, is solved inside the loss);
- $\tau=AU^*-b$: truncation error of the Laplacian rows and of the flux rows;
- $\rho=\operatorname{mean}_i\sum_j|\hat w_{ij}|\,\|\hat y_j\|^{\mu}$, $\mu=$ `deg`: weighted $\ell_1$ norm, eq. (20) of Davydov & Schaback.

The exact solution of the test problem is used only for testing.

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
import lfd
torch.manual_seed(0)

## 1. Upper bound: free weights on one node set (no network)

One trainable number per (row, stencil node) of the node set $X^{(2)}$ (uniform). This is the best the loss can reach on that node set.

In [ ]:
problem = lfd.TP1
geom = lfd.Geometry(problem, 2, 'uniform')
free = lfd.FreeCorrection(geom)
hist_free = lfd.train(free, [geom], steps=2000, nsol=32, lr=1e-3, log=250)
r0, r1 = lfd.solve(lfd.Baseline(), geom), lfd.solve(free, geom)
print(f"exact test-problem solution: rms baseline {r0['rms']:.3e} (sigma {r0['sigma']:.1f}),"
      f" free weights {r1['rms']:.3e} (sigma {r1['sigma']:.1f}), ratio {r1['rms'] / r0['rms']:.3f}")

## 2. Network: stencil geometry → correction $v$

Permutation-equivariant network (one for the Laplacian rows, one for the flux rows), trained on the levels [2, 3] (uniform and Halton) and tested on all levels.

In [ ]:
train_levels = [2, 3]
model = lfd.NetCorrection()
geoms = [lfd.Geometry(problem, m, nt) for m in train_levels for nt in ('uniform', 'halton')]
print('unknowns per training node set:', [g.N for g in geoms])
hist = lfd.train(model, geoms, steps=600, nsol=16, lr=1e-3, log=50)
torch.save(model.state_dict(), f'{problem.name}_net.pt')

In [ ]:
it = [h[0] for h in hist]; V = np.array([[t[0] for t in h[1]] for h in hist])
plt.figure(figsize=(6, 3.5))
for j, g in enumerate(geoms):
    plt.semilogy(it, V[:, j], 'o-', label=f'm = {g.m}, {g.ntype}')
plt.axhline(1, color='k', lw=0.5); plt.xlabel('step'); plt.ylabel(r'validation $\|E\|^2/\|E_0\|^2$')
plt.legend(); plt.grid(alpha=0.3); plt.title('Manufactured solutions: error relative to the baseline'); plt.show()

## 3. Test Problem 1: baseline (PHS $r^5$) against the learned weights
Errors over the interior nodes; `sigma` estimates $\|A^{-1}\|_\infty$.

In [ ]:
levels = range(0, 5)
res = {nt: lfd.table(model, problem, levels, nt) for nt in ('uniform', 'halton')}
ref = {nt: lfd.phs_table(problem, levels, nt, betas=(7, 9)) for nt in ('uniform', 'halton')}

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, nt in zip(ax, ('uniform', 'halton')):
    N = [r[2]['Nint'] for r in res[nt]]
    a.loglog(N, [r[2]['rms'] for r in res[nt]], 'o-', label='baseline (PHS $r^5$)')
    a.loglog(N, [r[3]['rms'] for r in res[nt]], 's-', label='learned weights')
    a.loglog(N, ref[nt][7], 'k:', label='PHS $r^7$')
    a.loglog(N, ref[nt][9], 'k--', label='PHS $r^9$')
    a.set_xlabel('$N_{int}$'); a.set_ylabel('rms error'); a.set_title(f'{problem.name}, {nt} nodes, degree 4')
    a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.show()